In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict

print("=" * 80)
print("Preference-Based Selection vs Original Interleaving Analysis")
print("=" * 80)

Preference-Based Selection vs Original Interleaving Analysis


In [2]:
# ============================================================================
# 1. 读取数据
# ============================================================================
print("\n[1] Loading data files...")

# 读取结果
try:
    original = pd.read_csv('result_interleaved_with_text.csv')
    original['qid'] = original['qid'].astype(str)
    print(f"  ✓ Original interleaving result: {len(original)} rows")
    has_original = True
except FileNotFoundError:
    print(f"  ✗ File not found: result_interleaved_with_text.csv")
    has_original = False

try:
    preference_based = pd.read_csv('result_preference_based_with_text.csv')
    preference_based['qid'] = preference_based['qid'].astype(str)
    print(f"  ✓ Preference-based result: {len(preference_based)} rows")
    has_preference = True
except FileNotFoundError:
    print(f"  ✗ File not found: result_preference_based_with_text.csv")
    has_preference = False

# 读取preference
try:
    preference_df = pd.read_csv('preference.csv')
    preference_df['qid'] = preference_df['qid'].astype(str)
    print(f"  ✓ Preference data: {len(preference_df)} queries")
    has_pref = True
except FileNotFoundError:
    print(f"  ✗ File not found: preference.csv")
    has_pref = False

if not (has_preference and has_pref):
    print("\n❌ Missing required files. Exiting.")
    exit(1)



[1] Loading data files...
  ✓ Original interleaving result: 387 rows
  ✓ Preference-based result: 387 rows
  ✓ Preference data: 43 queries


In [3]:
# ============================================================================
# 2. 基础统计
# ============================================================================
print("\n" + "=" * 80)
print("[2] Basic Statistics")
print("=" * 80)

# Preference分布
preference_map = preference_df.set_index('qid')['preference'].to_dict()
pref_counts = preference_df['preference'].value_counts()

print("\nPreference Distribution:")
for pref, count in pref_counts.items():
    print(f"  {pref:20s}: {count:3d} queries")

# 新结果的标签分布
print("\nPreference-Based Result - Origin Label Distribution:")
new_label_dist = preference_based['origin_label'].value_counts()
for label, count in new_label_dist.items():
    print(f"  {label:20s}: {count:3d} docs")

if has_original:
    print("\nOriginal Interleaving Result - Origin Label Distribution:")
    orig_label_dist = original['origin_label'].value_counts()
    for label, count in orig_label_dist.items():
        print(f"  {label:20s}: {count:3d} docs")


[2] Basic Statistics

Preference Distribution:
  Neutral             :  21 queries
  PRF-Hurt            :  11 queries
  PRF-Benefit         :   9 queries
  Insufficient-Data   :   2 queries

Preference-Based Result - Origin Label Distribution:
  E2E-Only            :  99 docs
  Both                :  94 docs
  PRF-Only            :  81 docs
  A-Only              :  51 docs
  B-Only              :  51 docs
  Easy-Negative       :  11 docs

Original Interleaving Result - Origin Label Distribution:
  Both                : 166 docs
  A-Only              : 100 docs
  B-Only              : 100 docs
  Easy-Negative       :  21 docs


In [4]:
# ============================================================================
# 3. 按Preference类型分析
# ============================================================================
print("\n" + "=" * 80)
print("[3] Analysis by Preference Type")
print("=" * 80)

for pref_type in ['PRF-Hurt', 'PRF-Benefit', 'Neutral', 'Insufficient-Data']:
    qids = preference_df[preference_df['preference'] == pref_type]['qid'].tolist()
    
    if len(qids) == 0:
        continue
    
    print(f"\n{pref_type} ({len(qids)} queries)")
    print("-" * 60)
    
    # 新结果
    new_subset = preference_based[preference_based['qid'].isin(qids)]
    new_labels = new_subset['origin_label'].value_counts()
    
    print(f"  Preference-based result ({len(new_subset)} docs):")
    for label, count in new_labels.items():
        pct = count / len(new_subset) * 100
        print(f"    {label:20s}: {count:3d} ({pct:5.1f}%)")
    
    # 如果有原始结果，对比
    if has_original:
        orig_subset = original[original['qid'].isin(qids)]
        orig_labels = orig_subset['origin_label'].value_counts()
        
        print(f"\n  Original interleaving ({len(orig_subset)} docs):")
        for label, count in orig_labels.items():
            pct = count / len(orig_subset) * 100
            print(f"    {label:20s}: {count:3d} ({pct:5.1f}%)")
        
        # 如果是Neutral/Insufficient-Data，检查是否一致
        if pref_type in ['Neutral', 'Insufficient-Data']:
            if new_labels.to_dict() == orig_labels.to_dict():
                print(f"\n  ✓ Labels match perfectly!")
            else:
                print(f"\n  ⚠️ Labels differ (this might indicate an issue)")



[3] Analysis by Preference Type

PRF-Hurt (11 queries)
------------------------------------------------------------
  Preference-based result (99 docs):
    E2E-Only            :  99 (100.0%)

  Original interleaving (99 docs):
    Both                :  42 ( 42.4%)
    A-Only              :  26 ( 26.3%)
    B-Only              :  26 ( 26.3%)
    Easy-Negative       :   5 (  5.1%)

PRF-Benefit (9 queries)
------------------------------------------------------------
  Preference-based result (81 docs):
    PRF-Only            :  81 (100.0%)

  Original interleaving (81 docs):
    Both                :  30 ( 37.0%)
    A-Only              :  23 ( 28.4%)
    B-Only              :  23 ( 28.4%)
    Easy-Negative       :   5 (  6.2%)

Neutral (21 queries)
------------------------------------------------------------
  Preference-based result (189 docs):
    Both                :  79 ( 41.8%)
    A-Only              :  50 ( 26.5%)
    B-Only              :  50 ( 26.5%)
    Easy-Negative      

In [5]:
# ============================================================================
# 4. 文档重叠分析 (PRF-Hurt 和 PRF-Benefit)
# ============================================================================
print("\n" + "=" * 80)
print("[4] Document Overlap Analysis")
print("=" * 80)

if has_original:
    # 读取原始排序数据（如果需要）
    try:
        df_colbert = pd.read_csv('df_colbert_deduped.csv')
        df_colbert_prf = pd.read_csv('df_colbert_prf_deduped.csv')
        df_colbert['qid'] = df_colbert['qid'].astype(str)
        df_colbert_prf['qid'] = df_colbert_prf['qid'].astype(str)
        has_ranking_data = True
    except:
        has_ranking_data = False
    
    for pref_type in ['PRF-Hurt', 'PRF-Benefit']:
        qids = preference_df[preference_df['preference'] == pref_type]['qid'].tolist()
        
        if len(qids) == 0:
            continue
        
        print(f"\n{pref_type} ({len(qids)} queries)")
        print("-" * 60)
        
        # 获取实际选择的文档
        actual_docs = preference_based[preference_based['qid'].isin(qids)]
        
        # 获取原始interleaving的文档
        original_docs = original[original['qid'].isin(qids)]
        
        # 按查询分析
        overlap_stats = []
        
        for qid in qids:
            actual_qid = set(actual_docs[actual_docs['qid'] == qid]['docno'])
            orig_qid = set(original_docs[original_docs['qid'] == qid]['docno'])
            
            overlap = len(actual_qid & orig_qid)
            unique_to_actual = len(actual_qid - orig_qid)
            unique_to_orig = len(orig_qid - actual_qid)
            
            overlap_stats.append({
                'qid': qid,
                'overlap': overlap,
                'unique_to_actual': unique_to_actual,
                'unique_to_orig': unique_to_orig,
                'total_actual': len(actual_qid),
                'total_orig': len(orig_qid)
            })
        
        overlap_df = pd.DataFrame(overlap_stats)
        
        print(f"  Average overlap: {overlap_df['overlap'].mean():.2f} / 9 docs")
        print(f"  Average unique to preference-based: {overlap_df['unique_to_actual'].mean():.2f}")
        print(f"  Average unique to original interleaving: {overlap_df['unique_to_orig'].mean():.2f}")
        
        # 如果有原始排序数据，分析Both的比例
        if has_ranking_data:
            both_count = 0
            total_docs = 0
            
            for qid in qids:
                actual_qid_docs = actual_docs[actual_docs['qid'] == qid]['docno'].tolist()
                
                e2e_docs = set(df_colbert[df_colbert['qid'] == qid]['docno'])
                prf_docs = set(df_colbert_prf[df_colbert_prf['qid'] == qid]['docno'])
                
                for doc in actual_qid_docs:
                    total_docs += 1
                    if doc in e2e_docs and doc in prf_docs:
                        both_count += 1
            
            if total_docs > 0:
                both_pct = both_count / total_docs * 100
                print(f"\n  Of the selected docs:")
                print(f"    {both_count}/{total_docs} ({both_pct:.1f}%) are in BOTH E2E and PRF rankings")
                print(f"    {total_docs - both_count}/{total_docs} ({100-both_pct:.1f}%) are unique to one system")


[4] Document Overlap Analysis

PRF-Hurt (11 queries)
------------------------------------------------------------
  Average overlap: 6.45 / 9 docs
  Average unique to preference-based: 2.55
  Average unique to original interleaving: 2.55

  Of the selected docs:
    96/99 (97.0%) are in BOTH E2E and PRF rankings
    3/99 (3.0%) are unique to one system

PRF-Benefit (9 queries)
------------------------------------------------------------
  Average overlap: 6.67 / 9 docs
  Average unique to preference-based: 2.33
  Average unique to original interleaving: 2.33

  Of the selected docs:
    77/81 (95.1%) are in BOTH E2E and PRF rankings
    4/81 (4.9%) are unique to one system


In [6]:
# ============================================================================
# 5. 查询级别详细对比 (前10个有变化的查询)
# ============================================================================
if has_original:
    print("\n" + "=" * 80)
    print("[5] Query-Level Comparison (Queries with Changes)")
    print("=" * 80)
    
    changed_queries = []
    
    for qid in preference_based['qid'].unique():
        pref_type = preference_map.get(qid, 'Unknown')
        
        # 只看Neutral/Insufficient-Data应该一致的查询
        if pref_type not in ['Neutral', 'Insufficient-Data']:
            # PRF-Hurt 和 PRF-Benefit 本来就会变化
            continue
        
        new_qid = preference_based[preference_based['qid'] == qid]
        orig_qid = original[original['qid'] == qid]
        
        new_labels = new_qid['origin_label'].value_counts().to_dict()
        orig_labels = orig_qid['origin_label'].value_counts().to_dict()
        
        if new_labels != orig_labels:
            changed_queries.append({
                'qid': qid,
                'preference': pref_type,
                'original': orig_labels,
                'new': new_labels
            })
    
    if len(changed_queries) > 0:
        print(f"\n⚠️ Found {len(changed_queries)} Neutral/Insufficient-Data queries with label changes:")
        print("   (This should be 0 if the algorithm is working correctly)\n")
        
        for item in changed_queries[:10]:
            print(f"  Query {item['qid']} ({item['preference']}):")
            print(f"    Original: {item['original']}")
            print(f"    New:      {item['new']}")
            print()
    else:
        print(f"\n✓ All {len([q for q in preference_based['qid'].unique() if preference_map.get(q) in ['Neutral', 'Insufficient-Data']])} Neutral/Insufficient-Data queries have consistent labels!")



[5] Query-Level Comparison (Queries with Changes)

✓ All 23 Neutral/Insufficient-Data queries have consistent labels!


In [8]:
#============================================================================
# 6. 总结
# ============================================================================
print("\n" + "=" * 80)
print("[6] Summary")
print("=" * 80)

print(f"""
Strategy Overview:
  - PRF-Hurt queries use ColBERT E2E Top-9
  - PRF-Benefit queries use ColBERT-PRF Top-9
  - Neutral/Insufficient-Data queries use Team Draft Interleaving

Total Documents: {len(preference_based)}
Total Queries: {preference_based['qid'].nunique()}

Label Distribution in Preference-Based Result:
""")

for label, count in new_label_dist.items():
    pct = count / len(preference_based) * 100
    print(f"  {label:20s}: {count:3d} ({pct:5.1f}%)")

# 检查A-Only和B-Only的平衡性
neutral_qids = preference_df[preference_df['preference'].isin(['Neutral', 'Insufficient-Data'])]['qid'].tolist()
neutral_subset = preference_based[preference_based['qid'].isin(neutral_qids)]
neutral_labels = neutral_subset['origin_label'].value_counts()

a_only = neutral_labels.get('A-Only', 0)
b_only = neutral_labels.get('B-Only', 0)

print(f"\nBalance Check for Neutral Queries:")
print(f"  A-Only: {a_only}")
print(f"  B-Only: {b_only}")

if a_only == b_only:
    print(f"  ✓ Perfect balance!")
elif abs(a_only - b_only) <= 1:
    print(f"  ✓ Nearly balanced (difference of 1 is acceptable)")
else:
    print(f"  ⚠️ Imbalanced by {abs(a_only - b_only)}")

print("\n" + "=" * 80)
print("Analysis Complete!")
print("=" * 80)


[6] Summary

Strategy Overview:
  - PRF-Hurt queries use ColBERT E2E Top-9
  - PRF-Benefit queries use ColBERT-PRF Top-9
  - Neutral/Insufficient-Data queries use Team Draft Interleaving

Total Documents: 387
Total Queries: 43

Label Distribution in Preference-Based Result:

  E2E-Only            :  99 ( 25.6%)
  Both                :  94 ( 24.3%)
  PRF-Only            :  81 ( 20.9%)
  A-Only              :  51 ( 13.2%)
  B-Only              :  51 ( 13.2%)
  Easy-Negative       :  11 (  2.8%)

Balance Check for Neutral Queries:
  A-Only: 51
  B-Only: 51
  ✓ Perfect balance!

Analysis Complete!
